### Импорты

In [41]:
import json
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score
from catboost import CatBoostClassifier, save_model

### Загрузка Json датасета

In [6]:
with open("dataset_train.json", "r", encoding="utf8") as f:
    train_data = json.load(f)

print(type(train_data))
print(len(train_data))
print(train_data[0])

<class 'list'>
4826
{'accountId': 1497, 'isCommercial': True, 'address': 'Краснодарский край, р-н Мостовский, пгт Мостовской, ул Колхозная, д. 37 1', 'buildingType': 'Частный', 'roomsCount': 1, 'residentsCount': 1, 'consumption': {'1': 3484, '2': 2824, '3': 3035, '4': 3597, '5': 2664, '6': 3874, '7': 3270, '8': 3714, '9': 3124, '10': 2981, '11': 2923, '12': 2216}}


### Превращаем Json в таблицу

In [8]:
train_df = pd.json_normalize(train_data)
train_df.columns = [
    col.replace("consumption.", "consumption_")
    for col in train_df.columns
]
print(train_df.shape)
display(train_df.head())

(4826, 19)


,accountId,isCommercial,address,buildingType,roomsCount,residentsCount,consumption_1,consumption_2,consumption_3,consumption_4,consumption_5,consumption_6,consumption_7,consumption_8,consumption_9,consumption_10,consumption_11,consumption_12,totalArea
0,1497,True,"Краснодарский край, р-н Мостовский, пгт Мостов...",Частный,1.0,1.0,3484.0,2824.0,3035.0,3597.0,2664.0,3874.0,3270.0,3714.0,3124.0,2981.0,2923.0,2216.0,NaN
1,1509,True,"Краснодарский край, р-н Усть-Лабинский, х Брат...",Частный,1.0,1.0,3756.0,1580.0,3191.0,2931.0,793.0,721.0,680.0,581.0,426.0,440.0,1206.0,2377.0,NaN
2,1674,True,"Краснодарский край, р-н Тбилисский, ст-ца Тбил...",Частный,3.0,2.0,1543.0,1075.0,2344.0,1125.0,1045.0,914.0,1731.0,1867.0,977.0,1178.0,1494.0,1495.0,80.5
3,1955,True,"Краснодарский край, р-н Анапский, с Сукко, ул ...",Частный,5.0,1.0,5564.0,6201.0,5364.0,4031.0,5452.0,NaN,NaN,NaN,NaN,590.0,4089.0,5408.0,908.0
4,1960,True,"Краснодарский край, р-н Анапский, село Витязев...",Частный,3.0,3.0,631.0,616.0,439.0,562.0,4723.0,9719.0,16720.0,20134.0,11680.0,3270.0,1215.0,726.0,NaN


In [51]:
print("Типы данных:")
print(train_df.dtypes)
print(train_df["isCommercial"].value_counts(dropna=False))
print()

print("Пропуски:")
print(train_df.isna().sum())

Типы данных:
accountId           int64
isCommercial         bool
address               str
buildingType          str
roomsCount        float64
residentsCount    float64
consumption_1     float64
consumption_2     float64
consumption_3     float64
consumption_4     float64
consumption_5     float64
consumption_6     float64
consumption_7     float64
consumption_8     float64
consumption_9     float64
consumption_10    float64
consumption_11    float64
consumption_12    float64
totalArea         float64
dtype: object
isCommercial
False    2935
True     1891
Name: count, dtype: int64

Пропуски:
accountId            0
isCommercial         0
address              0
buildingType         0
roomsCount         483
residentsCount     667
consumption_1      274
consumption_2      272
consumption_3      274
consumption_4      224
consumption_5      101
consumption_6      236
consumption_7      238
consumption_8      242
consumption_9      243
consumption_10     275
consumption_11     274
consumptio

### Отделяем признаки и target

In [10]:
y = train_df["isCommercial"]
x = train_df.drop(columns=["accountId", "address", "isCommercial"])

In [11]:
print(x.shape)
print(y.shape)
print(x.columns.tolist())

(4826, 16)
(4826,)
['buildingType', 'roomsCount', 'residentsCount', 'consumption_1', 'consumption_2', 'consumption_3', 'consumption_4', 'consumption_5', 'consumption_6', 'consumption_7', 'consumption_8', 'consumption_9', 'consumption_10', 'consumption_11', 'consumption_12', 'totalArea']


### Определяем категориальные признаки

In [16]:
categorical_features = x.select_dtypes(include=["str"]).columns.tolist()
print(categorial_features)

['buildingType']


### Делим на train и validate и test

In [86]:
x_train, x_temp, y_train, y_temp = train_test_split(
    x,
    y,
    test_size = 0.3,
    random_state = 42,
    stratify = y,
)

x_valid, x_test, y_valid, y_test = train_test_split(
    x_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp,
)

print(x_train.shape)
print(x_test.shape)
print(x_valid.shape)

(3378, 16)
(724, 16)
(724, 16)


### Функция для доп признаков

In [87]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    cons_cols = [f"consumption_{i}" for i in range(1, 13)]

    # Пропуски
    df["roomsCount_missing"] = df["roomsCount"].isna().astype(int)
    df["residentsCount_missing"] = df["residentsCount"].isna().astype(int)
    df["totalArea_missing"] = df["totalArea"].isna().astype(int)
    df["consumption_missing_count"] = df[cons_cols].isna().sum(axis=1)

    # Базовые агрегаты
    #df["cons_sum"] = df[cons_cols].sum(axis=1)
    #df["cons_mean"] = df[cons_cols].mean(axis=1)
    #df["cons_max"] = df[cons_cols].max(axis=1)
    #df["cons_min"] = df[cons_cols].min(axis=1)
    #df["cons_std"] = df[cons_cols].std(axis=1)
    #df["cons_range"] = df["cons_max"] - df["cons_min"]

    # Сезонность
    #winter_cols = ["consumption_12", "consumption_1", "consumption_2"]
    #summer_cols = ["consumption_6", "consumption_7", "consumption_8"]

    #df["winter_mean"] = df[winter_cols].mean(axis=1)
    #df["summer_mean"] = df[summer_cols].mean(axis=1)
    #df["winter_summer_diff"] = df["winter_mean"] - df["summer_mean"]
    #df["winter_summer_ratio"] = df["winter_mean"] / (df["summer_mean"] + 1e-6)

    # Нормированные признаки
    #df["cons_sum_per_area"] = df["cons_sum"] / (df["totalArea"] + 1e-6)
    #df["cons_mean_per_resident"] = df["cons_mean"] / (df["residentsCount"] + 1e-6)

    return df

### Добавляем доп признаки в данные

In [88]:
#x_train_fe = add_features(x_train)
#x_valid_fe = add_features(x_valid)
#x_test_fe = add_features(x_test)

### Определяем категориальные признаки

In [89]:
categorical_features = x_train.select_dtypes(include=["str"]).columns.tolist()
print(categorial_features)

['buildingType']


### Создаем catboost model

In [73]:
model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=5,
    l2_leaf_reg=20,
    auto_class_weights="Balanced",
    loss_function="Logloss",
    eval_metric="F1",
    random_seed=42,
    verbose=100,
    od_type="Iter",
    od_wait=100,
    use_best_model=True,
)

### Обучаем модель

In [92]:
model.fit(
    x_train,
    y_train,
    cat_features = categorical_features,
    eval_set=(x_valid, y_valid)
)

0:	learn: 0.6355971	test: 0.5945927	best: 0.5945927 (0)	total: 122ms	remaining: 6m 5s
100:	learn: 0.7002068	test: 0.6538793	best: 0.6584755 (98)	total: 13.3s	remaining: 6m 20s
200:	learn: 0.7223842	test: 0.6588693	best: 0.6623340 (148)	total: 27.4s	remaining: 6m 22s
300:	learn: 0.7425845	test: 0.6746639	best: 0.6746639 (283)	total: 42.9s	remaining: 6m 24s
400:	learn: 0.7745021	test: 0.6793286	best: 0.6809063 (326)	total: 57.9s	remaining: 6m 15s
500:	learn: 0.7925040	test: 0.7052834	best: 0.7090868 (489)	total: 1m 12s	remaining: 6m 2s
600:	learn: 0.8164623	test: 0.6928429	best: 0.7099021 (508)	total: 1m 26s	remaining: 5m 45s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7099020814
bestIteration = 508

Shrink model to first 509 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=5, eval_metric='F1', iterations=3000, l2_leaf_reg=20, learning_rate=0.03, loss_function='Logloss', od_type='Iter', od_wait=100, random_seed=42, use_best_model=True, verbose=100)

### Созраняем в файл

In [94]:
model.save_model("catboost_model.cbm")

### Делаем предикты

In [96]:
#y_pred = model.predict(x_test)
#y_proba = model.predict_proba(x_test)
y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)[:, 1]

### Подбираем Threshold

In [46]:
results = []

thresholds = np.arange(0.1, 0.91, 0.05)

for thr in thresholds:
    y_pred_thr = (y_proba >= thr).astype(int)

    results.append({
        "threshold": thr,
        "f1_true": f1_score(y_valid, y_pred_thr),
        "precision_true": precision_score(y_valid, y_pred_thr),
        "recall_true": recall_score(y_valid, y_pred_thr),
    })

threshold_df = pd.DataFrame(results).sort_values("f1_true", ascending=False)
display(threshold_df)

,threshold,f1_true,precision_true,recall_true
8,0.50,0.658940,0.619938,0.703180
7,0.45,0.649547,0.567282,0.759717
6,0.40,0.643258,0.533800,0.809187
9,0.55,0.627820,0.670683,0.590106
5,0.35,0.626788,0.495885,0.851590
4,0.30,0.609844,0.461818,0.897527
3,0.25,0.594235,0.432956,0.946996
2,0.20,0.580576,0.415902,0.961131
1,0.15,0.575884,0.407953,0.978799
10,0.60,0.575000,0.700508,0.487633


### Метрики

In [97]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print()

print("Classification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[297 143]
 [105 179]]

Classification Report:
              precision    recall  f1-score   support

       False       0.74      0.68      0.71       440
        True       0.56      0.63      0.59       284

    accuracy                           0.66       724
   macro avg       0.65      0.65      0.65       724
weighted avg       0.67      0.66      0.66       724

